# AOPC Scoring — All Models

**Run cells 1 → 2 → 3 → 4 → 5 → 6 in order.**

Computes AOPC (Area Over the Perturbation Curve) for:
- `efficientnetb4` / `gradcam`
- `efficientnetb4` / `lime`
- `resnet50` / `gradcam`
- `resnet50` / `lime`
- `vit_b16` / `attention_rollout`

**Wait for ViT heatmaps to exist before running the vit_b16 row.**
The notebook skips any combo where heatmaps aren't on disk yet.

### What AOPC measures
- **Deletion**: start with full image, progressively mask out the most important
  pixels (replaced with dataset mean). Measures how fast model confidence drops.
  Higher AOPC = heatmap correctly identified important regions.
- **Insertion**: start with blurred image, progressively reveal most important
  pixels. Measures how fast confidence recovers. Higher = better.
- Both are computed at steps of 1% of pixels, up to 50%.
- Saved per image to `results/scores/aopc_{model}_{method}.csv`


In [ ]:
# Cell 1: Mount Drive + Clone/Pull Repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, importlib

REPO_URL = 'https://github.com/rao-shreyashree/diabetic-retinopathy-xai.git'
REPO_DIR = '/content/diabetic-retinopathy-xai'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

for folder in ['datasets', 'xai', 'utils']:
    init = os.path.join(REPO_DIR, folder, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
print('Repo ready')


In [ ]:
# Cell 2: Install Dependencies
os.system('pip install timm -q')
print('Done')


In [ ]:
# Cell 3: Imports + Paths
import os, sys, json, glob, csv
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from scipy.ndimage import gaussian_filter

REPO_DIR = '/content/diabetic-retinopathy-xai'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import timm
import torchvision.models as tv_models
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

PROJECT_ROOT = '/content/drive/MyDrive/Projects/diabetic-retinopathy-xai'
CKPT_DIR     = f'{PROJECT_ROOT}/results/checkpoints'
IMG_DIR      = '/content/drive/MyDrive/IDRiD/grading/images/test'
HEATMAP_DIR  = f'{PROJECT_ROOT}/results/heatmaps'
SCORES_DIR   = f'{PROJECT_ROOT}/results/scores'
PRED_CSV     = f'{PROJECT_ROOT}/results/heatmaps/predictions.csv'

os.makedirs(SCORES_DIR, exist_ok=True)

with open(os.path.join(REPO_DIR, 'test_image_ids.json')) as f:
    TEST_IDS = json.load(f)

pred_df = pd.read_csv(PRED_CSV)

# All model/method combos to score
# vit_b16 row is skipped automatically if heatmaps don't exist yet
COMBOS = [
    ('efficientnetb4', 'gradcam'),
    ('efficientnetb4', 'lime'),
    ('resnet50',       'gradcam'),
    ('resnet50',       'lime'),
    ('vit_b16',        'attention_rollout'),
]

print(f'{len(TEST_IDS)} test IDs')
print(f'Scores will be saved to: {SCORES_DIR}')


In [ ]:
# Cell 4: Model loader + AOPC implementation

NUM_CLASSES = 5

def resolve_checkpoint(ckpt_dir, pattern):
    matches = sorted(glob.glob(os.path.join(ckpt_dir, pattern)))
    if not matches:
        raise FileNotFoundError(f"No checkpoint matching '{pattern}' in {ckpt_dir}")
    clean = [m for m in matches if '(' not in os.path.basename(m)]
    return clean[0] if clean else matches[0]

def load_model(model_name, device):
    """Load model from checkpoint. Returns (model, transform, img_size)."""
    if model_name == 'efficientnetb4':
        model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=NUM_CLASSES)
        ckpt  = resolve_checkpoint(CKPT_DIR, 'efficientnet_b4_clean_best*.pth')
        img_size = 512
    elif model_name == 'resnet50':
        model = tv_models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        ckpt  = resolve_checkpoint(CKPT_DIR, 'resnet50_clean_best*.pth')
        img_size = 512
    elif model_name == 'vit_b16':
        model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
        ckpt  = resolve_checkpoint(CKPT_DIR, 'vit_b16_best*.pth')
        img_size = 224
    else:
        raise ValueError(f'Unknown model: {model_name}')

    state_dict = torch.load(ckpt, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return model, transform, img_size


def get_confidence(model, tensor, target_class, device):
    """Returns softmax probability for target_class."""
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)
    return probs[0, target_class].item()


def compute_aopc(model, transform, img_size, pil_img,
                 heatmap, target_class, device,
                 n_steps=50, mode='deletion'):
    """
    Computes AOPC for one image.

    mode='deletion': mask out most important pixels progressively
    mode='insertion': reveal most important pixels into blurred image

    Returns scalar AOPC score.
    """
    # Work at model input resolution for efficiency
    pil_resized = pil_img.resize((img_size, img_size), Image.BILINEAR)
    img_arr     = np.array(pil_resized, dtype=np.float32) / 255.0  # (H, W, 3)

    # Resize heatmap to match
    hm_tensor = torch.from_numpy(heatmap).unsqueeze(0).unsqueeze(0)
    hm_small  = F.interpolate(hm_tensor, size=(img_size, img_size),
                               mode='bilinear', align_corners=False)
    hm_small  = hm_small.squeeze().numpy()   # (img_size, img_size)

    # Rank all pixels by importance (highest first)
    flat_order = np.argsort(hm_small.ravel())[::-1]   # descending
    total_pixels = img_size * img_size
    step_size    = total_pixels // n_steps

    # Baselines
    mean_pixel = img_arr.mean(axis=(0, 1))             # dataset mean substitute
    blurred    = gaussian_filter(img_arr, sigma=10)    # for insertion baseline

    if mode == 'deletion':
        perturbed = img_arr.copy()
    else:
        perturbed = blurred.copy()

    # Original confidence
    orig_tensor = transform(pil_resized)
    f0 = get_confidence(model, orig_tensor, target_class, device)

    scores = []
    for step in range(1, n_steps + 1):
        pixel_indices = flat_order[: step * step_size]
        rows = pixel_indices // img_size
        cols = pixel_indices  % img_size

        if mode == 'deletion':
            perturbed[rows, cols] = mean_pixel
        else:
            perturbed[rows, cols] = img_arr[rows, cols]

        # Convert perturbed array back to PIL -> tensor
        pil_p  = Image.fromarray((perturbed * 255).clip(0, 255).astype(np.uint8))
        tensor = transform(pil_p)
        score  = get_confidence(model, tensor, target_class, device)
        scores.append(score)

    if mode == 'deletion':
        # AOPC_del = f0 - mean(scores): how much confidence drops
        aopc = f0 - np.mean(scores)
    else:
        # AOPC_ins = mean(scores) - f_blurred: how much confidence builds
        blurred_pil    = Image.fromarray((blurred * 255).clip(0, 255).astype(np.uint8))
        blurred_tensor = transform(blurred_pil)
        f_blur = get_confidence(model, blurred_tensor, target_class, device)
        aopc   = np.mean(scores) - f_blur

    return float(aopc)


def clean_id(image_id):
    return f'IDRiD_{int(image_id.replace("IDRiD_", ""))}'

def load_image(image_id, img_dir):
    num = int(image_id.replace('IDRiD_', ''))
    return Image.open(os.path.join(img_dir, f'IDRiD_{num:03d}.jpg')).convert('RGB')

def get_pred_grade(pred_df, image_id, model_name):
    cid = clean_id(image_id)
    row = pred_df[(pred_df['image_id'] == cid) & (pred_df['model'] == model_name)]
    if len(row) == 0:
        # vit_b16 predictions won't be in the CSV yet — fall back to predicted argmax
        return None
    return int(row['predicted_grade'].values[0])

print('AOPC functions ready')


In [ ]:
# Cell 5: Run AOPC for all combos
# Skips any combo where heatmap files don't exist yet.
# Resume-safe: if the output CSV already exists and has the right rows, skips.

import time

N_STEPS = 50   # 50 steps = 1% increments up to 50% of pixels

for model_name, method in COMBOS:
    heatmap_folder = os.path.join(HEATMAP_DIR, model_name, method)
    npy_files      = glob.glob(os.path.join(heatmap_folder, '*.npy'))

    if len(npy_files) == 0:
        print(f'\n[SKIP] {model_name}/{method} — no heatmaps found at {heatmap_folder}')
        continue

    out_csv = os.path.join(SCORES_DIR, f'aopc_{model_name}_{method}.csv')

    # Resume: load already-done image IDs
    done_ids = set()
    if os.path.exists(out_csv):
        existing = pd.read_csv(out_csv)
        done_ids = set(existing['image_id'].values)
        print(f'\n=== {model_name}/{method} === ({len(done_ids)} already done)')
    else:
        print(f'\n=== {model_name}/{method} ===')

    # Load model once per combo
    try:
        model, transform, img_size = load_model(model_name, DEVICE)
    except FileNotFoundError as e:
        print(f'  [SKIP] checkpoint not found: {e}')
        continue

    rows = []
    for i, image_id in enumerate(TEST_IDS):
        cid = clean_id(image_id)
        if cid in done_ids:
            print(f'  [{i+1:02d}/27] {cid} already done — skipping')
            continue

        # Load heatmap
        hm_path = os.path.join(heatmap_folder, f'{model_name}_{method}_{cid}.npy')
        if not os.path.exists(hm_path):
            print(f'  [{i+1:02d}/27] {cid} heatmap missing — skipping')
            continue

        try:
            heatmap = np.load(hm_path)
            pil_img = load_image(image_id, IMG_DIR)

            # Get predicted class to explain
            pred_grade = get_pred_grade(pred_df, image_id, model_name)
            if pred_grade is None:
                # For vit_b16 not in predictions.csv, run inference
                vit_transform = T.Compose([
                    T.Resize((img_size, img_size)), T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
                with torch.no_grad():
                    logits = model(vit_transform(pil_img).unsqueeze(0).to(DEVICE))
                pred_grade = logits.argmax().item()

            t0 = time.time()
            aopc_del = compute_aopc(model, transform, img_size, pil_img,
                                    heatmap, pred_grade, DEVICE,
                                    n_steps=N_STEPS, mode='deletion')
            aopc_ins = compute_aopc(model, transform, img_size, pil_img,
                                    heatmap, pred_grade, DEVICE,
                                    n_steps=N_STEPS, mode='insertion')
            elapsed = time.time() - t0

            rows.append({
                'image_id':        cid,
                'model':           model_name,
                'method':          method,
                'predicted_grade': pred_grade,
                'aopc_deletion':   round(aopc_del, 6),
                'aopc_insertion':  round(aopc_ins, 6),
            })
            print(f'  [{i+1:02d}/27] {cid} del={aopc_del:.4f} ins={aopc_ins:.4f} ({elapsed:.1f}s)')

        except Exception as e:
            print(f'  [{i+1:02d}/27] {cid} FAILED: {e}')

    # Append to CSV
    if rows:
        new_df = pd.DataFrame(rows)
        if os.path.exists(out_csv):
            new_df.to_csv(out_csv, mode='a', header=False, index=False)
        else:
            new_df.to_csv(out_csv, index=False)
        print(f'  Saved {len(rows)} rows -> {out_csv}')

    del model
    torch.cuda.empty_cache()

print('\nAll combos done!')


In [ ]:
# Cell 6: Summary — print mean AOPC scores per combo

print('AOPC Summary')
print('=' * 60)
print(f'{"Model/Method":<35} {"Del AOPC":>10} {"Ins AOPC":>10}')
print('-' * 60)

for model_name, method in COMBOS:
    csv_path = os.path.join(SCORES_DIR, f'aopc_{model_name}_{method}.csv')
    if not os.path.exists(csv_path):
        print(f'{model_name}/{method:<20} not computed yet')
        continue
    df = pd.read_csv(csv_path)
    mean_del = df['aopc_deletion'].mean()
    mean_ins = df['aopc_insertion'].mean()
    n        = len(df)
    label    = f'{model_name}/{method}'
    print(f'{label:<35} {mean_del:>10.4f} {mean_ins:>10.4f}  (n={n})')

print('=' * 60)
print(f'\nCSVs saved to: {SCORES_DIR}')
